# Housing Prices Prediction System

### Loading Modules 

In [6]:
# Standard modules
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# sklearn modules 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [7]:
# loading the dataset
melbourne_data = pd.read_csv(r"C:\Users\DT\Desktop\Practice\Data\melb_data\melb_data.csv") # converting the dataset into a pandas dataframe
melbourne_data.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


### Retrieving information about the Dataframe

In [8]:
print(melbourne_data.info()) # print information about the dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

### Splitting data into features and target

In [9]:
# splitting the dataset into features and target
X = melbourne_data.drop(['Price'], axis=1) # selects all features except the target variable 'Price'
y = melbourne_data.Price # selects the target variable 'Price'

### Splitting data into training and testing sets

In [10]:
# splitting the dataset into training and testing sets with 80% training and 20% testing with a random state of 42 for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

### Data Exploration

In [11]:
y_train.isna().sum() # check for missing values in the target variable of the training set

0

In [12]:
# exploring the X training data
X_train.head()

,Suburb,Address,Rooms,Type,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
12796,Mount Waverley,37 Pascall St,4,h,S,Jellis,16/09/2017,14.2,3149.0,4.0,2.0,2.0,695.0,160.0,1970.0,NaN,-37.86127,145.14271,Eastern Metropolitan,13366.0
9642,Mount Waverley,23 Baily St,3,h,S,Ray,17/06/2017,14.2,3149.0,3.0,1.0,2.0,810.0,NaN,NaN,Monash,-37.86838,145.14664,Eastern Metropolitan,13366.0
3207,Hawthorn,5/70 Power St,2,u,S,Jellis,25/02/2017,4.6,3122.0,2.0,1.0,1.0,82.0,NaN,NaN,Boroondara,-37.81800,145.02680,Southern Metropolitan,11308.0
1698,Carlton North,24/635 Drummond St,2,u,S,hockingstuart,27/06/2016,3.2,3054.0,2.0,1.0,1.0,0.0,76.0,1975.0,Yarra,-37.79020,144.97000,Northern Metropolitan,3106.0
761,Bentleigh,3 Somers St,4,h,S,Woodards,22/05/2016,13.0,3204.0,4.0,2.0,1.0,292.0,NaN,NaN,Glen Eira,-37.91480,145.02430,Southern Metropolitan,6795.0


In [13]:
X_train.describe().style.format("{:,.2f}") # provides a statistical summary of the numeric features in the training dataset, including count, mean, standard deviation, minimum, maximum, and quartiles for each feature.

,Rooms,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,"10,864.00","10,864.00","10,864.00","10,864.00","10,864.00","10,814.00","10,864.00","5,735.00","6,578.00","10,864.00","10,864.00","10,864.00"
mean,2.94,10.10,"3,105.38",2.91,1.53,1.61,524.16,154.00,"1,965.10",-37.81,144.99,"7,463.12"
std,0.96,5.89,91.58,0.97,0.69,0.96,"1,380.50",601.70,36.37,0.08,0.10,"4,379.16"
min,1.00,0.00,"3,000.00",0.00,0.00,0.00,0.00,0.00,"1,830.00",-38.18,144.43,249.00
25%,2.00,6.10,"3,046.00",2.00,1.00,1.00,175.00,92.00,"1,940.00",-37.86,144.93,"4,385.00"
50%,3.00,9.20,"3,084.00",3.00,1.00,2.00,435.00,126.00,"1,970.00",-37.80,145.00,"6,567.00"
75%,3.00,13.00,"3,147.00",3.00,2.00,2.00,650.25,175.00,"2,000.00",-37.76,145.06,"10,331.00"
max,10.00,48.10,"3,977.00",20.00,8.00,10.00,"75,100.00","44,515.00","2,018.00",-37.41,145.53,"21,650.00"


In [14]:
X_train.info() # this returns information about the training dataset, including the number of non-null entries, data types of each feature, and memory usage.

<class 'pandas.core.frame.DataFrame'>
Index: 10864 entries, 12796 to 7270
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         10864 non-null  object 
 1   Address        10864 non-null  object 
 2   Rooms          10864 non-null  int64  
 3   Type           10864 non-null  object 
 4   Method         10864 non-null  object 
 5   SellerG        10864 non-null  object 
 6   Date           10864 non-null  object 
 7   Distance       10864 non-null  float64
 8   Postcode       10864 non-null  float64
 9   Bedroom2       10864 non-null  float64
 10  Bathroom       10864 non-null  float64
 11  Car            10814 non-null  float64
 12  Landsize       10864 non-null  float64
 13  BuildingArea   5735 non-null   float64
 14  YearBuilt      6578 non-null   float64
 15  CouncilArea    9774 non-null   object 
 16  Lattitude      10864 non-null  float64
 17  Longtitude     10864 non-null  float64
 18  Regionna

### Missing Values

In [15]:
# checking for missing numbers
missing_values = X_train.isna().sum() # calculates the number of missing values in each feature of the training dataset and stores it in a variable called 'missing_values'.
missing_values[missing_values > 0] # displays the features with missing values and their corresponding counts

Car               50
BuildingArea    5129
YearBuilt       4286
CouncilArea     1090
dtype: int64

### Categorical Variables

In [16]:
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns.tolist() # selects the categorical features from the training dataset and stores them in a variable called 'categorical_features'.
X_train[categorical_features].nunique()  # count unique values for each categorical feature

Suburb           306
Address        10726
Type               3
Method             5
SellerG          249
Date              58
CouncilArea       33
Regionname         8
dtype: int64

### Handling Missing Values and Categorical Variables
Since the unique values per each of the columns varies greatly, we will use difference encoding methods on the data

selecting the categorical features with more than 10 unique values for target encoding and the categorical features with 10 or fewer unique values for one-hot encoding, as well as the features to be dropped from the training dataset and the numeric features from the training dataset.

In [17]:
drop_col = ['Address', 'Postcode', 'Date']
# categorical_features = X_train.select_dtypes(include=['object']).columns # selects the categorical features from the training dataset and stores them in a variable called 'categorical_features'.

categorical_features = [
    col for col in categorical_features
    if col not in drop_col
] # filters out the features in 'drop_col' from the list of categorical features and stores the remaining categorical features in a variable called 'categorical_features'.

target_encode_col = [
    col for col in categorical_features
    if X_train[col].nunique() > 10
] # filters the categorical features to include only those with more than 10 unique values and stores them in a variable called 'target_encode_col'.

onehot_encode_col = [
    col for col in categorical_features
    if X_train[col].nunique() <= 10
] # filters the categorical features to include only those with 10 or fewer unique values and stores them in a variable called 'onehot_encode_col'.

numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist() # selects the numeric features from the training dataset and stores them in a variable called 'numeric_features'.
numeric_features = [col for col in numeric_features if col not in drop_col] # filters out the features in 'drop_col' from the list of numeric features and stores the remaining numeric features in a variable called 'numeric_features'.

In [18]:
categorical_features

['Suburb', 'Type', 'Method', 'SellerG', 'CouncilArea', 'Regionname']

In [19]:
# Combines all selected features into a single list
total_features = target_encode_col + onehot_encode_col + numeric_features # combines the target encoded, one-hot encoded, and numeric features into a single list called 'total_features'.

# Selects the features in 'total_features' from the training and testing datasets
X_train_updated = X_train[total_features].copy() # creates a copy of the training dataset with only the selected features and stores it in a variable called 'X_train_updated'.
X_test_updated = X_test[total_features].copy() # creates a copy of the testing dataset with only the selected features and stores it in a variable called 'X_test_updated'.

In [20]:
print(X_train_updated.info()) # print information about the updated training dataset

<class 'pandas.core.frame.DataFrame'>
Index: 10864 entries, 12796 to 7270
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         10864 non-null  object 
 1   SellerG        10864 non-null  object 
 2   CouncilArea    9774 non-null   object 
 3   Type           10864 non-null  object 
 4   Method         10864 non-null  object 
 5   Regionname     10864 non-null  object 
 6   Rooms          10864 non-null  int64  
 7   Distance       10864 non-null  float64
 8   Bedroom2       10864 non-null  float64
 9   Bathroom       10864 non-null  float64
 10  Car            10814 non-null  float64
 11  Landsize       10864 non-null  float64
 12  BuildingArea   5735 non-null   float64
 13  YearBuilt      6578 non-null   float64
 14  Lattitude      10864 non-null  float64
 15  Longtitude     10864 non-null  float64
 16  Propertycount  10864 non-null  float64
dtypes: float64(10), int64(1), object(6)
memory usage: 1.

In [21]:
# numerical transformer pipeline for preprocessing numeric features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # replaces missing values in numeric features with the median value of the respective feature
    ('scaler', StandardScaler()) # standardizes the numeric features by removing the mean and scaling to unit variance
])

In [22]:
# categorical transformer pipeline for preprocessing categorical features
onehot_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # replaces missing values in categorical features with the most frequent value of the respective feature
    ('onehot', OneHotEncoder(handle_unknown='ignore')), # applies one-hot encoding to the categorical features, ignoring any unknown categories during transformation
])

In [23]:
target_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # replaces missing values in categorical features with the most frequent value of the respective feature  
    ('target', TargetEncoder()) # applies target encoding to the categorical features, encoding them based on the mean of the target variable for each category
])

In [24]:
# preprocessing both numerical and categorical features using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numeric_features), # applies the numerical transformer to the numeric features
        ('onehot', onehot_transformer, onehot_encode_col),
        ('target', target_transformer, target_encode_col)
    ]
)

In [25]:
model = LinearRegression() # creates an instance of the LinearRegression model and stores it in a variable called 'model'.

In [26]:
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor), 
        ('model', model)
    ]) # creates a pipeline that first preprocesses the data using the 'preprocessor' and then fits the 'model' to the preprocessed data.

In [27]:
pipeline = pipeline.fit(X_train_updated, y_train)

In [28]:
y_pred = pipeline.predict(X_test_updated) # uses the fitted pipeline to make predictions on the updated testing dataset and stores the predicted values in a variable called 'y_pred'
mae = mean_absolute_error(y_test, y_pred) # calculates the mean absolute error between the actual and predicted values of the target variable in the testing dataset and stores it in a variable called 'mae'.
print("The mean square error is:", mae)

The mean square error is: 240134.05722810057


In [29]:
# the mean square error
# it is a measure of how close the predicted values are to the actual values. 
# It is calculated by taking the average of the squared differences between the predicted and actual values. 
# A lower mean square error indicates better model performance.
mse  = mean_squared_error(y_test, y_pred)
print("The mean square error: ", mse)

The mean square error:  128999114728.29285


In [30]:
# the root mean square error
# it is the square root of the mean square error.
rmse = np.sqrt(mse)
print("The root mean square error: ", rmse)

The root mean square error:  359164.4675191198


In [31]:
# the coefficient of determination
r_square = r2_score(y_test, y_pred)
print("The r squared value: ", r_square)

The r squared value:  0.6752401422734676


In [32]:
n = X_test.shape[0]
p = X_test.shape[1]

adj_r2 = 1 - (1 - r_square)* (n -1) / (n - p - 1)
print("Adjusted R square value is:", adj_r2)

Adjusted R square value is: 0.6728300505649218


Adjusted  
R2 asks a very simple question: "Did this new feature actually help the model predict better, or is it just taking up space?" If we add a strong, useful feature (like horsepower), the predictive power increases enough to beat the penalty, and our Adjusted R2 goes up

a parsimonious model is one that achieves strong predictive power using the simplest possible structure. The Akaike Information Criterion (AIC) and Bayesian Information Criterion (BIC) are statistical metrics designed to evaluate this exact balance.

When we build several different models to predict the exact same thing, we need a way to declare a mathematical winner. Both AIC (Akaike Information Criterion) and BIC (Bayesian Information Criterion) act as referees based on two strict rules:

Reward accuracy: The closer the predictions are to reality, the better the score.

Penalize bloat: Every time we add a new feature (variable) to the model, they add a penalty

In [33]:
import statsmodels.formula.api as smf
 
train_df = X_train_updated.copy()
train_df['Price'] = y_train
formula = 'Price ~ ' + ' + '.join(X_train_updated.columns) # this is the formula that we will use to predict the price of the house based on the features selected above
# which is the same as Price ~ Rooms + Distance + Postcode + Bedroom2 + Bathroom + Car + Landsize + BuildingArea + YearBuilt
model = smf.ols(formula=formula, data=train_df).fit() # this is the model that we will use to predict the price of the house based on the features selected above

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.747
Model:                            OLS   Adj. R-squared:                  0.719
Method:                 Least Squares   F-statistic:                     26.98
Date:                Wed, 01 Jul 2026   Prob (F-statistic):               0.00
Time:                        12:33:06   Log-Likelihood:                -70198.
No. Observations:                4959   AIC:                         1.414e+05
Df Residuals:                    4469   BIC:                         1.446e+05
Df Model:                         489                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

In [34]:
print("AIC:", model.aic) # AIC (Akaike Information Criterion) is a measure of the relative quality of a statistical model for a given set of data.
#It is used to compare different models and select the best one. A lower AIC value indicates a better-fitting model.



print("BIC:", model.bic) # BIC (Bayesian Information Criterion) is another measure of the relative quality of a statistical model. 
#It penalizes more complex models more heavily than AIC. A lower BIC value indicates a better-fitting model.

AIC: 141375.76836491274
BIC: 144565.1584642924
